# Mframapa AI - PM2.5 Model Training

This notebook trains an XGBoost model to predict PM2.5 air quality across Africa using satellite and weather data.

Before running, make sure you have:
- Uploaded super_training_dataset.csv to Google Drive
- Enabled GPU runtime (Runtime > Change runtime type > GPU)

In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
import os
import shutil
from google.colab import drive
from datetime import datetime

drive.mount('/content/drive')

# paths to your google drive folders
CHECKPOINT_DIR = '/content/drive/MyDrive/mframapa/checkpoints'
MODEL_DIR = '/content/drive/MyDrive/mframapa/models'
DATA_PATH = '/content/drive/MyDrive/mframapa/super_training_dataset.csv'

# clear old checkpoints so we start fresh
if os.path.exists(CHECKPOINT_DIR):
    shutil.rmtree(CHECKPOINT_DIR)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print('setup complete')

In [ ]:
def load_checkpoint(name):
    """load a saved checkpoint if it exists"""
    path = f'{CHECKPOINT_DIR}/{name}.pkl'
    if os.path.exists(path):
        with open(path, 'rb') as f:
            print(f'loaded checkpoint: {name}')
            return pickle.load(f)
    return None

def save_checkpoint(name, data):
    """save a checkpoint so we can resume later if needed"""
    path = f'{CHECKPOINT_DIR}/{name}.pkl'
    with open(path, 'wb') as f:
        pickle.dump(data, f)
    print(f'saved checkpoint: {name}')

## 1. Load and Clean Data

In [ ]:
print('loading data...')
df = pd.read_csv(DATA_PATH)
print(f'loaded {len(df):,} rows')

# show what the pm2.5 values look like before we clean them
print('\npm2.5 stats before cleaning:')
print(df['pm25'].describe())

In [ ]:
# remove outliers - values above 500 or below/equal to 0 are likely sensor errors
before_count = len(df)
df = df[(df['pm25'] > 0) & (df['pm25'] <= 500)]
after_count = len(df)

removed = before_count - after_count
print(f'removed {removed:,} outliers ({100*removed/before_count:.1f}%)')
print(f'remaining: {after_count:,} rows')

print('\npm2.5 stats after cleaning:')
print(df['pm25'].describe())

## 2. Feature Engineering

In [ ]:
print('running feature engineering...')

df['datetime'] = pd.to_datetime(df['datetime'])

# create region groupings - we use 5 degree bands for latitude and longitude
# this helps us understand how the model performs across different areas
df['lat_region'] = (df['lat'] // 5).astype(int)
df['lon_region'] = (df['lon'] // 5).astype(int)
df['region'] = df['lat_region'].astype(str) + '_' + df['lon_region'].astype(str)

# cyclical encoding for time features
# this helps the model understand that hour 23 is close to hour 0
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# african seasonal flags
# harmattan is the dusty dry season from december to february
# rainy season typically runs may through september
df['is_harmattan'] = df['month'].isin([12, 1, 2]).astype(int)
df['is_rainy'] = df['month'].isin([5, 6, 7, 8, 9]).astype(int)

# drop location column to prevent the model from just memorizing station names
# we keep lat and lon so it can learn geographic patterns
df = df.drop(columns=['location', 'datetime'], errors='ignore')

# fill any missing values with the median of that column
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

print(f'feature engineering complete. shape: {df.shape}')

## 3. Prepare Features and Target

In [ ]:
TARGET = 'pm25'

# exclude the target and helper columns from features
EXCLUDE = ['pm25', 'region', 'lat_region', 'lon_region', 'day_of_week']
FEATURES = [c for c in df.columns if c not in EXCLUDE]

print(f'target: {TARGET}')
print(f'features ({len(FEATURES)}): {FEATURES}')

X = df[FEATURES]
y = df[TARGET]

print(f'\nX shape: {X.shape}')
print(f'pm2.5 range: {y.min():.1f} to {y.max():.1f}')
print(f'pm2.5 mean: {y.mean():.1f}')

## 4. Train Test Split

In [ ]:
# split into 80% train, 10% validation, 10% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f'train: {len(X_train):,} samples')
print(f'val:   {len(X_val):,} samples')
print(f'test:  {len(X_test):,} samples')

## 5. Train XGBoost Model

In [ ]:
print('training xgboost model...')

# these parameters are tuned for good accuracy while avoiding overfitting
params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    
    # tree structure - deeper trees can capture more complex patterns
    'max_depth': 12,
    'min_child_weight': 10,
    
    # sampling - using most of the data but not all helps prevent overfitting
    'subsample': 0.9,
    'colsample_bytree': 0.9,
    'colsample_bylevel': 0.9,
    
    # regularization - keeps the model from getting too complex
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'gamma': 0.05,
    
    # slower learning rate means more trees but better accuracy
    'learning_rate': 0.02,
    
    # use gpu for faster training
    'tree_method': 'hist',
    'device': 'cuda',
    
    'random_state': 42,
    'verbosity': 1
}

# create the data matrices xgboost needs
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)

evallist = [(dtrain, 'train'), (dval, 'eval')]

print('training with early stopping, this may take 10-30 minutes...')

model = xgb.train(
    params,
    dtrain,
    num_boost_round=3000,
    evals=evallist,
    early_stopping_rounds=100,
    verbose_eval=50
)

save_checkpoint('xgb_model_v2', {'model': model, 'params': params})
print('training complete')

## 6. Evaluate Model

In [ ]:
y_pred = model.predict(dtest)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print('test set metrics:')
print(f'  rmse: {rmse:.2f} ug/m3')
print(f'  mae:  {mae:.2f} ug/m3')
print(f'  r2:   {r2:.3f}')

In [ ]:
# calculate how often we get the health category right
# this is more practical than just rmse since users care about the category

def pm25_to_category(pm25):
    """convert pm2.5 value to aqi health category"""
    if pm25 <= 12:
        return 'good'
    elif pm25 <= 35.4:
        return 'moderate'
    elif pm25 <= 55.4:
        return 'unhealthy for sensitive groups'
    elif pm25 <= 150.4:
        return 'unhealthy'
    elif pm25 <= 250.4:
        return 'very unhealthy'
    else:
        return 'hazardous'

y_test_cat = [pm25_to_category(v) for v in y_test]
y_pred_cat = [pm25_to_category(v) for v in y_pred]

category_accuracy = sum([a == b for a, b in zip(y_test_cat, y_pred_cat)]) / len(y_test)
print(f'aqi category accuracy: {100*category_accuracy:.1f}%')

## 7. Feature Importance

In [ ]:
importance = model.get_score(importance_type='gain')
importance_df = pd.DataFrame([
    {'feature': k, 'importance': v} 
    for k, v in importance.items()
]).sort_values('importance', ascending=False)

print('top 10 most important features:')
print(importance_df.head(10).to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(15), x='importance', y='feature', hue='feature', palette='viridis', legend=False)
plt.title('top 15 feature importances')
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/feature_importance_v2.png', dpi=150)
plt.show()

## 8. Prediction Scatter Plot

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.1, s=1)
plt.plot([0, 300], [0, 300], 'r--', label='perfect prediction')
plt.xlabel('actual pm2.5 (ug/m3)')
plt.ylabel('predicted pm2.5 (ug/m3)')
plt.title(f'prediction vs actual (r2 = {r2:.3f})')
plt.xlim(0, 300)
plt.ylim(0, 300)
plt.legend()
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/prediction_scatter_v2.png', dpi=150)
plt.show()

## 9. Save Model

In [ ]:
model_path = f'{MODEL_DIR}/universal_african_model.json'
model.save_model(model_path)
print(f'model saved to: {model_path}')

# save the feature list and metrics so we know what this model expects
config = {
    'features': FEATURES,
    'target': TARGET,
    'metrics': {
        'rmse': float(rmse),
        'mae': float(mae),
        'r2': float(r2),
        'category_accuracy': float(category_accuracy)
    },
    'training_date': datetime.now().isoformat(),
    'samples': len(df),
    'version': 2
}

with open(f'{MODEL_DIR}/features.json', 'w') as f:
    json.dump(config, f, indent=2)

print('\ntraining complete.')
print(f'rmse: {rmse:.2f} | mae: {mae:.2f} | r2: {r2:.3f}')
print(f'category accuracy: {100*category_accuracy:.1f}%')
print('\ndownload the model and place it in backend/models/')